In [ ]:
import os, sys
import torch
import pandas as pd
import numpy as np
from google.colab import drive

drive.mount('/content/drive')

sys.path.insert(0, '/content/drive/MyDrive/CSE720/code')
from config import Config

cfg = Config()
print("cfg.base_dir:", cfg.base_dir)

Mounted at /content/drive
cfg.base_dir: /content/drive/MyDrive/CSE720


In [ ]:
%%writefile dcgan.py
"""
Self-contained DCGAN baseline — no dependency on other files in this pipeline.

Usage (separate Colab cells):
    !python dcgan.py --mode train
    !python dcgan.py --mode train --resume        # if a session cuts off mid-training
    !python dcgan.py --mode evaluate --checkpoint checkpoints/dcgan/latest.pth

DCGAN generates images from random noise - it has no mechanism to target a specific domain
or a specific source image. Because of this, comparing a generated image to any single real
image via PSNR/SSIM/MSE is not really a meaningful pointwise comparison (there is no
correspondence between the two). Those numbers are still computed here for completeness and
consistency with the other baselines, but FID (which compares distributions, not individual
images) is the metric that actually means something for DCGAN.

Uses the SAME seeded train/val/test split logic as the EyeGAN pipeline (same root_dir,
seed=42, ratios 0.8/0.1), so results are comparable / usable with stats_tests.py.
"""
import os
import csv
import random
import argparse
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import numpy as np
from PIL import Image
import torchvision.transforms as transforms
from skimage.metrics import peak_signal_noise_ratio as sk_psnr
from skimage.metrics import structural_similarity as sk_ssim
from torchvision.utils import save_image

CHECKPOINT_DIR = '/content/drive/MyDrive/CSE720/checkpoints/dcgan'
RESULTS_DIR = '/content/drive/MyDrive/CSE720/results/dcgan'
DATASET_PATH = '/content/drive/MyDrive/CSE720/EyeGAN'

IMG_SIZE = 64
BATCH_SIZE = 8
NZ = 100
LR = 2e-4
NUM_EPOCHS = 100
SAVE_FREQ = 5

TRAIN_RATIO = 0.8
VAL_RATIO = 0.1
FLIP_PROB = 0.3
SEED = 42


# ============================================================
# Dataset (inlined, same logic/seed as EyeGAN's dataset.py)
# ============================================================

class RetinalDataset(Dataset):
    """
    Expects root_dir/<domain_name>/*.png|jpg with one subfolder per class.
    Split is stratified per domain and seeded for reproducibility.
    """

    def __init__(self, root_dir, img_size=64, mode='train',
                 train_ratio=0.8, val_ratio=0.1, flip_prob=0.3, seed=42):
        assert mode in ('train', 'val', 'test')
        self.root_dir = root_dir
        self.mode = mode
        self.img_size = img_size

        self.domains = sorted(
            d.strip() for d in os.listdir(root_dir)
            if os.path.isdir(os.path.join(root_dir, d))
        )
        self.domain_to_label = {d: i for i, d in enumerate(self.domains)}
        self.label_to_domain = {i: d for d, i in self.domain_to_label.items()}

        self.image_paths = []
        self.labels = []

        rng = random.Random(seed)

        for domain in self.domains:
            domain_path = os.path.join(root_dir, domain)
            images = sorted(
                f for f in os.listdir(domain_path)
                if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp'))
            )
            rng.shuffle(images)

            n = len(images)
            n_train = int(n * train_ratio)
            n_val = int(n * (train_ratio + val_ratio))

            if mode == 'train':
                selected = images[:n_train]
            elif mode == 'val':
                selected = images[n_train:n_val]
            else:
                selected = images[n_val:]

            for f in selected:
                self.image_paths.append(os.path.join(domain_path, f))
                self.labels.append(self.domain_to_label[domain])

        tfms = [transforms.Resize((img_size, img_size))]
        if mode == 'train' and flip_prob > 0:
            tfms.append(transforms.RandomHorizontalFlip(p=flip_prob))
        tfms += [
            transforms.ToTensor(),
            transforms.Normalize([0.5] * 3, [0.5] * 3),
        ]
        self.transform = transforms.Compose(tfms)

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        path = self.image_paths[idx]
        img = Image.open(path).convert('RGB')
        img = self.transform(img)
        return img, self.labels[idx], path


# ============================================================
# Model
# ============================================================

class Generator(nn.Module):
    def __init__(self, nz=100, ngf=64, nc=3):
        super().__init__()
        self.nz = nz
        self.main = nn.Sequential(
            nn.ConvTranspose2d(nz, ngf * 8, 4, 1, 0, bias=False),
            nn.BatchNorm2d(ngf * 8), nn.ReLU(True),
            nn.ConvTranspose2d(ngf * 8, ngf * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf * 4), nn.ReLU(True),
            nn.ConvTranspose2d(ngf * 4, ngf * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf * 2), nn.ReLU(True),
            nn.ConvTranspose2d(ngf * 2, ngf, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf), nn.ReLU(True),
            nn.ConvTranspose2d(ngf, nc, 4, 2, 1, bias=False),
            nn.Tanh(),
        )

    def forward(self, z):
        return self.main(z.view(-1, self.nz, 1, 1))


class Discriminator(nn.Module):
    def __init__(self, ndf=64, nc=3):
        super().__init__()
        self.main = nn.Sequential(
            nn.Conv2d(nc, ndf, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf, ndf * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 2), nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf * 2, ndf * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 4), nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf * 4, ndf * 8, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 8), nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf * 8, 1, 4, 1, 0, bias=False),
            nn.Sigmoid(),
        )

    def forward(self, x):
        return self.main(x).view(-1)


def weights_init(m):
    name = m.__class__.__name__
    if 'Conv' in name:
        nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif 'BatchNorm' in name:
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0)


def denorm(t):
    return (t.clamp(-1, 1) + 1) / 2


def train(args):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    os.makedirs(CHECKPOINT_DIR, exist_ok=True)

    train_ds = RetinalDataset(DATASET_PATH, IMG_SIZE, 'train', TRAIN_RATIO, VAL_RATIO, FLIP_PROB, SEED)
    loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, drop_last=True)

    G, D = Generator(NZ).to(device), Discriminator().to(device)
    G.apply(weights_init)
    D.apply(weights_init)

    opt_g = optim.Adam(G.parameters(), lr=LR, betas=(0.5, 0.999))
    opt_d = optim.Adam(D.parameters(), lr=LR, betas=(0.5, 0.999))
    criterion = nn.BCELoss()

    start_epoch = 0
    latest_path = os.path.join(CHECKPOINT_DIR, 'latest.pth')
    if args.resume and os.path.exists(latest_path):
        ckpt = torch.load(latest_path, map_location=device)
        G.load_state_dict(ckpt['G'])
        D.load_state_dict(ckpt['D'])
        opt_g.load_state_dict(ckpt['opt_g'])
        opt_d.load_state_dict(ckpt['opt_d'])
        start_epoch = ckpt['epoch'] + 1
        print(f'Resumed from epoch {start_epoch}')

    for epoch in range(start_epoch, NUM_EPOCHS):
        for step, (real, _, _) in enumerate(loader):
            real = real.to(device)
            b = real.size(0)
            real_labels = torch.ones(b, device=device)
            fake_labels = torch.zeros(b, device=device)

            opt_d.zero_grad()
            loss_d_real = criterion(D(real), real_labels)
            noise = torch.randn(b, NZ, device=device)
            fake = G(noise)
            loss_d_fake = criterion(D(fake.detach()), fake_labels)
            loss_d = loss_d_real + loss_d_fake
            loss_d.backward()
            opt_d.step()

            opt_g.zero_grad()
            loss_g = criterion(D(fake), real_labels)
            loss_g.backward()
            opt_g.step()

            if step % 50 == 0:
                print(f'epoch {epoch+1}/{NUM_EPOCHS} step {step}/{len(loader)} '
                      f'd_loss={loss_d.item():.4f} g_loss={loss_g.item():.4f}')

        if (epoch + 1) % SAVE_FREQ == 0 or (epoch + 1) == NUM_EPOCHS:
            torch.save({
                'epoch': epoch, 'G': G.state_dict(), 'D': D.state_dict(),
                'opt_g': opt_g.state_dict(), 'opt_d': opt_d.state_dict(),
            }, latest_path)
            print(f'Saved checkpoint at epoch {epoch+1}')

    print('DCGAN training finished.')


def evaluate(args):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    os.makedirs(RESULTS_DIR, exist_ok=True)
    real_dir = os.path.join(RESULTS_DIR, 'real')
    fake_dir = os.path.join(RESULTS_DIR, 'fake')
    os.makedirs(real_dir, exist_ok=True)
    os.makedirs(fake_dir, exist_ok=True)

    ckpt = torch.load(args.checkpoint, map_location=device)
    G = Generator(NZ).to(device)
    G.load_state_dict(ckpt['G'])
    G.eval()

    test_ds = RetinalDataset(DATASET_PATH, IMG_SIZE, 'test', TRAIN_RATIO, VAL_RATIO, flip_prob=0.0, seed=SEED)
    loader = DataLoader(test_ds, batch_size=1, shuffle=False)

    rows = []
    with torch.no_grad():
        for idx, (real, label, _) in enumerate(loader):
            real = real.to(device)
            noise = torch.randn(1, NZ, device=device)
            fake = G(noise)

            save_image(denorm(real[0]), os.path.join(real_dir, f'{idx:05d}.png'))
            save_image(denorm(fake[0]), os.path.join(fake_dir, f'{idx:05d}.png'))

            real_np = denorm(real[0]).cpu().numpy().transpose(1, 2, 0)
            fake_np = denorm(fake[0]).cpu().numpy().transpose(1, 2, 0)

            mse = float(np.mean((real_np - fake_np) ** 2))
            psnr_val = sk_psnr(real_np, fake_np, data_range=1.0)
            ssim_val = sk_ssim(real_np, fake_np, data_range=1.0, channel_axis=2)

            rows.append({
                'image_idx': idx, 'source_domain': label.item(), 'target_domain': -1,
                'psnr': psnr_val, 'ssim': ssim_val, 'mse': mse,
            })

    csv_path = os.path.join(RESULTS_DIR, 'per_image_metrics.csv')
    with open(csv_path, 'w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=['image_idx', 'source_domain', 'target_domain', 'psnr', 'ssim', 'mse'])
        writer.writeheader()
        writer.writerows(rows)

    print(f'Saved {csv_path}')
    print(f'Real images: {real_dir}  |  Fake images: {fake_dir}')
    print(f'FID: python analysis.py --mode fid --real {real_dir} --fake {fake_dir}')


if __name__ == '__main__':
    p = argparse.ArgumentParser()
    p.add_argument('--mode', choices=['train', 'evaluate'], required=True)
    p.add_argument('--resume', action='store_true')
    p.add_argument('--checkpoint', type=str, default=os.path.join(CHECKPOINT_DIR, 'latest.pth'))
    args = p.parse_args()

    if args.mode == 'train':
        train(args)
    else:
        evaluate(args)


Writing dcgan.py


In [ ]:
%%writefile cyclegan.py
"""
Self-contained CycleGAN baseline — no dependency on other files in this pipeline.

Usage (separate Colab cells):
    !python cyclegan.py --mode train
    !python cyclegan.py --mode train --resume
    !python cyclegan.py --mode evaluate --checkpoint checkpoints/cyclegan/latest.pth

Uses the SAME seeded train/val/test split logic as the EyeGAN pipeline (same root_dir,
seed=42, ratios 0.8/0.1), filtered down to two single domains, so results are comparable /
usable with stats_tests.py.
"""
import os
import csv
import random
import argparse
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import numpy as np
from PIL import Image
import torchvision.transforms as transforms
from skimage.metrics import peak_signal_noise_ratio as sk_psnr
from skimage.metrics import structural_similarity as sk_ssim
from torchvision.utils import save_image

CHECKPOINT_DIR = '/content/drive/MyDrive/CSE720/checkpoints/cyclegan'
RESULTS_DIR = '/content/drive/MyDrive/CSE720/results/cyclegan'
DATASET_PATH = '/content/drive/MyDrive/CSE720/EyeGAN'

SOURCE_DOMAIN = 'Healthy'
TARGET_DOMAIN = 'Diabetic Retinopathy'  # FIXED: real folder name uses a space, not an underscore (see 00_Shared_Modules_Setup.ipynb dataset_statistics output)

IMG_SIZE = 64
BATCH_SIZE = 4
LR = 2e-4
NUM_EPOCHS = 100
SAVE_FREQ = 5
LAMBDA_CYCLE = 10.0
LAMBDA_IDENTITY = 5.0

TRAIN_RATIO = 0.8
VAL_RATIO = 0.1
FLIP_PROB = 0.3
SEED = 42


# ============================================================
# Dataset (inlined: RetinalDataset + single-domain subset)
# ============================================================

class RetinalDataset(Dataset):
    """
    Expects root_dir/<domain_name>/*.png|jpg with one subfolder per class.
    Split is stratified per domain and seeded for reproducibility.
    """

    def __init__(self, root_dir, img_size=64, mode='train',
                 train_ratio=0.8, val_ratio=0.1, flip_prob=0.3, seed=42):
        assert mode in ('train', 'val', 'test')
        self.root_dir = root_dir
        self.mode = mode
        self.img_size = img_size

        self.domains = sorted(
            d.strip() for d in os.listdir(root_dir)
            if os.path.isdir(os.path.join(root_dir, d))
        )
        self.domain_to_label = {d: i for i, d in enumerate(self.domains)}
        self.label_to_domain = {i: d for d, i in self.domain_to_label.items()}

        self.image_paths = []
        self.labels = []

        rng = random.Random(seed)

        for domain in self.domains:
            domain_path = os.path.join(root_dir, domain)
            images = sorted(
                f for f in os.listdir(domain_path)
                if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp'))
            )
            rng.shuffle(images)

            n = len(images)
            n_train = int(n * train_ratio)
            n_val = int(n * (train_ratio + val_ratio))

            if mode == 'train':
                selected = images[:n_train]
            elif mode == 'val':
                selected = images[n_train:n_val]
            else:
                selected = images[n_val:]

            for f in selected:
                self.image_paths.append(os.path.join(domain_path, f))
                self.labels.append(self.domain_to_label[domain])

        tfms = [transforms.Resize((img_size, img_size))]
        if mode == 'train' and flip_prob > 0:
            tfms.append(transforms.RandomHorizontalFlip(p=flip_prob))
        tfms += [
            transforms.ToTensor(),
            transforms.Normalize([0.5] * 3, [0.5] * 3),
        ]
        self.transform = transforms.Compose(tfms)

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        path = self.image_paths[idx]
        img = Image.open(path).convert('RGB')
        img = self.transform(img)
        return img, self.labels[idx], path


class DomainSubset(RetinalDataset):
    """RetinalDataset filtered down to a single domain (by folder name)."""

    def __init__(self, root_dir, domain_name, img_size=64, mode='train',
                 train_ratio=0.8, val_ratio=0.1, flip_prob=0.3, seed=42):
        super().__init__(root_dir, img_size, mode, train_ratio, val_ratio, flip_prob, seed)
        keep = [i for i, l in enumerate(self.labels) if self.label_to_domain[l] == domain_name]
        self.image_paths = [self.image_paths[i] for i in keep]
        self.labels = [self.labels[i] for i in keep]


# ============================================================
# Model
# ============================================================

class ResidualBlock(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.block = nn.Sequential(
            nn.ReflectionPad2d(1), nn.Conv2d(dim, dim, 3), nn.InstanceNorm2d(dim), nn.ReLU(inplace=True),
            nn.ReflectionPad2d(1), nn.Conv2d(dim, dim, 3), nn.InstanceNorm2d(dim),
        )

    def forward(self, x):
        return x + self.block(x)


class Generator(nn.Module):
    def __init__(self, in_channels=3, out_channels=3, n_residual_blocks=6):
        super().__init__()
        model = [nn.ReflectionPad2d(3), nn.Conv2d(in_channels, 64, 7),
                 nn.InstanceNorm2d(64), nn.ReLU(inplace=True)]
        in_f, out_f = 64, 128
        for _ in range(2):
            model += [nn.Conv2d(in_f, out_f, 3, stride=2, padding=1),
                      nn.InstanceNorm2d(out_f), nn.ReLU(inplace=True)]
            in_f, out_f = out_f, out_f * 2
        for _ in range(n_residual_blocks):
            model += [ResidualBlock(in_f)]
        out_f = in_f // 2
        for _ in range(2):
            model += [nn.ConvTranspose2d(in_f, out_f, 3, stride=2, padding=1, output_padding=1),
                      nn.InstanceNorm2d(out_f), nn.ReLU(inplace=True)]
            in_f, out_f = out_f, out_f // 2
        model += [nn.ReflectionPad2d(3), nn.Conv2d(64, out_channels, 7), nn.Tanh()]
        self.model = nn.Sequential(*model)

    def forward(self, x):
        return self.model(x)


class Discriminator(nn.Module):
    def __init__(self, in_channels=3):
        super().__init__()

        def block(in_f, out_f, normalize=True):
            layers = [nn.Conv2d(in_f, out_f, 4, stride=2, padding=1)]
            if normalize:
                layers.append(nn.InstanceNorm2d(out_f))
            layers.append(nn.LeakyReLU(0.2, inplace=True))
            return layers

        self.model = nn.Sequential(
            *block(in_channels, 64, normalize=False),
            *block(64, 128), *block(128, 256), *block(256, 512),
            nn.Conv2d(512, 1, 4, padding=1),
        )

    def forward(self, x):
        return self.model(x)


class ReplayBuffer:
    def __init__(self, max_size=50):
        self.max_size = max_size
        self.data = []

    def push_and_pop(self, data):
        out = []
        for element in data:
            element = element.unsqueeze(0)
            if len(self.data) < self.max_size:
                self.data.append(element)
                out.append(element)
            elif random.random() > 0.5:
                i = random.randint(0, self.max_size - 1)
                out.append(self.data[i].clone())
                self.data[i] = element
            else:
                out.append(element)
        return torch.cat(out)


def weights_init(m):
    name = m.__class__.__name__
    if 'Conv' in name:
        nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif 'BatchNorm' in name:
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0)


def denorm(t):
    return (t.clamp(-1, 1) + 1) / 2


def train(args):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    os.makedirs(CHECKPOINT_DIR, exist_ok=True)

    ds_a = DomainSubset(DATASET_PATH, SOURCE_DOMAIN, IMG_SIZE, 'train', TRAIN_RATIO, VAL_RATIO, FLIP_PROB, SEED)
    ds_b = DomainSubset(DATASET_PATH, TARGET_DOMAIN, IMG_SIZE, 'train', TRAIN_RATIO, VAL_RATIO, FLIP_PROB, SEED)
    loader_a = DataLoader(ds_a, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
    loader_b = DataLoader(ds_b, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)

    G_AtoB, G_BtoA = Generator().to(device), Generator().to(device)
    D_A, D_B = Discriminator().to(device), Discriminator().to(device)
    for m in (G_AtoB, G_BtoA, D_A, D_B):
        m.apply(weights_init)

    opt_g = optim.Adam(list(G_AtoB.parameters()) + list(G_BtoA.parameters()), lr=LR, betas=(0.5, 0.999))
    opt_d_a = optim.Adam(D_A.parameters(), lr=LR, betas=(0.5, 0.999))
    opt_d_b = optim.Adam(D_B.parameters(), lr=LR, betas=(0.5, 0.999))

    criterion_gan = nn.MSELoss()
    criterion_cycle = nn.L1Loss()
    criterion_identity = nn.L1Loss()

    buffer_a, buffer_b = ReplayBuffer(), ReplayBuffer()

    with torch.no_grad():
        dummy = torch.randn(1, 3, IMG_SIZE, IMG_SIZE).to(device)
        patch_size = D_A(dummy).shape[2]

    start_epoch = 0
    latest_path = os.path.join(CHECKPOINT_DIR, 'latest.pth')
    if args.resume and os.path.exists(latest_path):
        ckpt = torch.load(latest_path, map_location=device)
        G_AtoB.load_state_dict(ckpt['G_AtoB'])
        G_BtoA.load_state_dict(ckpt['G_BtoA'])
        D_A.load_state_dict(ckpt['D_A'])
        D_B.load_state_dict(ckpt['D_B'])
        opt_g.load_state_dict(ckpt['opt_g'])
        opt_d_a.load_state_dict(ckpt['opt_d_a'])
        opt_d_b.load_state_dict(ckpt['opt_d_b'])
        start_epoch = ckpt['epoch'] + 1
        print(f'Resumed from epoch {start_epoch}')

    for epoch in range(start_epoch, NUM_EPOCHS):
        n_steps = min(len(loader_a), len(loader_b))
        for step, (batch_a, batch_b) in enumerate(zip(loader_a, loader_b)):
            real_a = batch_a[0].to(device)
            real_b = batch_b[0].to(device)
            b = real_a.size(0)
            valid = torch.ones(b, 1, patch_size, patch_size, device=device)
            fake = torch.zeros(b, 1, patch_size, patch_size, device=device)

            opt_g.zero_grad()
            loss_id_a = criterion_identity(G_BtoA(real_a), real_a) * LAMBDA_IDENTITY
            loss_id_b = criterion_identity(G_AtoB(real_b), real_b) * LAMBDA_IDENTITY
            fake_b = G_AtoB(real_a)
            loss_gan_ab = criterion_gan(D_B(fake_b), valid)
            fake_a = G_BtoA(real_b)
            loss_gan_ba = criterion_gan(D_A(fake_a), valid)
            recov_a = G_BtoA(fake_b)
            loss_cycle_a = criterion_cycle(recov_a, real_a) * LAMBDA_CYCLE
            recov_b = G_AtoB(fake_a)
            loss_cycle_b = criterion_cycle(recov_b, real_b) * LAMBDA_CYCLE
            loss_g = loss_id_a + loss_id_b + loss_gan_ab + loss_gan_ba + loss_cycle_a + loss_cycle_b
            loss_g.backward()
            opt_g.step()

            opt_d_a.zero_grad()
            loss_d_a_real = criterion_gan(D_A(real_a), valid)
            fake_a_buf = buffer_a.push_and_pop(fake_a)
            loss_d_a_fake = criterion_gan(D_A(fake_a_buf.detach()), fake)
            loss_d_a = (loss_d_a_real + loss_d_a_fake) / 2
            loss_d_a.backward()
            opt_d_a.step()

            opt_d_b.zero_grad()
            loss_d_b_real = criterion_gan(D_B(real_b), valid)
            fake_b_buf = buffer_b.push_and_pop(fake_b)
            loss_d_b_fake = criterion_gan(D_B(fake_b_buf.detach()), fake)
            loss_d_b = (loss_d_b_real + loss_d_b_fake) / 2
            loss_d_b.backward()
            opt_d_b.step()

            if step % 50 == 0:
                print(f'epoch {epoch+1}/{NUM_EPOCHS} step {step}/{n_steps} '
                      f'g_loss={loss_g.item():.4f} d_loss={(loss_d_a+loss_d_b).item():.4f}')

        if (epoch + 1) % SAVE_FREQ == 0 or (epoch + 1) == NUM_EPOCHS:
            torch.save({
                'epoch': epoch, 'G_AtoB': G_AtoB.state_dict(), 'G_BtoA': G_BtoA.state_dict(),
                'D_A': D_A.state_dict(), 'D_B': D_B.state_dict(),
                'opt_g': opt_g.state_dict(), 'opt_d_a': opt_d_a.state_dict(), 'opt_d_b': opt_d_b.state_dict(),
            }, latest_path)
            print(f'Saved checkpoint at epoch {epoch+1}')

    print('CycleGAN training finished.')


def evaluate(args):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    os.makedirs(RESULTS_DIR, exist_ok=True)
    real_dir = os.path.join(RESULTS_DIR, 'real')
    fake_dir = os.path.join(RESULTS_DIR, 'fake')
    os.makedirs(real_dir, exist_ok=True)
    os.makedirs(fake_dir, exist_ok=True)

    ckpt = torch.load(args.checkpoint, map_location=device)
    G_AtoB = Generator().to(device)
    G_AtoB.load_state_dict(ckpt['G_AtoB'])
    G_AtoB.eval()

    test_ds = DomainSubset(DATASET_PATH, SOURCE_DOMAIN, IMG_SIZE, 'test', TRAIN_RATIO, VAL_RATIO, flip_prob=0.0, seed=SEED)
    loader = DataLoader(test_ds, batch_size=1, shuffle=False)

    rows = []
    with torch.no_grad():
        for idx, (real, label, _) in enumerate(loader):
            real = real.to(device)
            fake_img = G_AtoB(real)

            save_image(denorm(real[0]), os.path.join(real_dir, f'{idx:05d}.png'))
            save_image(denorm(fake_img[0]), os.path.join(fake_dir, f'{idx:05d}.png'))

            real_np = denorm(real[0]).cpu().numpy().transpose(1, 2, 0)
            fake_np = denorm(fake_img[0]).cpu().numpy().transpose(1, 2, 0)

            mse = float(np.mean((real_np - fake_np) ** 2))
            psnr_val = sk_psnr(real_np, fake_np, data_range=1.0)
            ssim_val = sk_ssim(real_np, fake_np, data_range=1.0, channel_axis=2)

            rows.append({
                'image_idx': idx, 'source_domain': SOURCE_DOMAIN, 'target_domain': TARGET_DOMAIN,
                'psnr': psnr_val, 'ssim': ssim_val, 'mse': mse,
            })

    csv_path = os.path.join(RESULTS_DIR, 'per_image_metrics.csv')
    with open(csv_path, 'w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=['image_idx', 'source_domain', 'target_domain', 'psnr', 'ssim', 'mse'])
        writer.writeheader()
        writer.writerows(rows)

    print(f'Saved {csv_path}')
    print(f'Real images: {real_dir}  |  Fake images: {fake_dir}')
    print(f'FID: python analysis.py --mode fid --real {real_dir} --fake {fake_dir}')


if __name__ == '__main__':
    p = argparse.ArgumentParser()
    p.add_argument('--mode', choices=['train', 'evaluate'], required=True)
    p.add_argument('--resume', action='store_true')
    p.add_argument('--checkpoint', type=str, default=os.path.join(CHECKPOINT_DIR, 'latest.pth'))
    args = p.parse_args()

    if args.mode == 'train':
        train(args)
    else:
        evaluate(args)


Writing cyclegan.py


In [ ]:
%%writefile pix2pix.py
"""
Self-contained Pix2Pix baseline — no dependency on other files in this pipeline.

Usage (separate Colab cells):
    !python pix2pix.py --mode train
    !python pix2pix.py --mode train --resume
    !python pix2pix.py --mode evaluate --checkpoint checkpoints/pix2pix/latest.pth

Note: Pix2Pix needs paired images. There is no true pairing available (no same-eye
healthy/DR pairs exist in this dataset), so PairedDataset (below) pairs images
positionally as a stand-in. This is intentional, not a bug - it is what produces the
L1-driven blurring failure mode discussed in the paper.

Uses the SAME seeded train/val/test split logic as the EyeGAN pipeline (same root_dir,
seed=42, ratios 0.8/0.1), so results are comparable / usable with stats_tests.py.
"""
import os
import csv
import random
import argparse
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import numpy as np
from PIL import Image
import torchvision.transforms as transforms
from skimage.metrics import peak_signal_noise_ratio as sk_psnr
from skimage.metrics import structural_similarity as sk_ssim
from torchvision.utils import save_image

CHECKPOINT_DIR = '/content/drive/MyDrive/CSE720/checkpoints/pix2pix'
RESULTS_DIR = '/content/drive/MyDrive/CSE720/results/pix2pix'
DATASET_PATH = '/content/drive/MyDrive/CSE720/EyeGAN'

SOURCE_DOMAIN = 'Healthy'
TARGET_DOMAIN = 'Diabetic Retinopathy'  # FIXED: real folder name uses a space, not an underscore (see 00_Shared_Modules_Setup.ipynb dataset_statistics output)

IMG_SIZE = 64
BATCH_SIZE = 8
LR = 2e-4
NUM_EPOCHS = 100
SAVE_FREQ = 5
LAMBDA_L1 = 100.0

TRAIN_RATIO = 0.8
VAL_RATIO = 0.1
FLIP_PROB = 0.3
SEED = 42


# ============================================================
# Dataset (inlined: RetinalDataset + single-domain subset + positional pairing)
# ============================================================

class RetinalDataset(Dataset):
    """
    Expects root_dir/<domain_name>/*.png|jpg with one subfolder per class.
    Split is stratified per domain and seeded for reproducibility.
    """

    def __init__(self, root_dir, img_size=64, mode='train',
                 train_ratio=0.8, val_ratio=0.1, flip_prob=0.3, seed=42):
        assert mode in ('train', 'val', 'test')
        self.root_dir = root_dir
        self.mode = mode
        self.img_size = img_size

        self.domains = sorted(
            d.strip() for d in os.listdir(root_dir)
            if os.path.isdir(os.path.join(root_dir, d))
        )
        self.domain_to_label = {d: i for i, d in enumerate(self.domains)}
        self.label_to_domain = {i: d for d, i in self.domain_to_label.items()}

        self.image_paths = []
        self.labels = []

        rng = random.Random(seed)

        for domain in self.domains:
            domain_path = os.path.join(root_dir, domain)
            images = sorted(
                f for f in os.listdir(domain_path)
                if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp'))
            )
            rng.shuffle(images)

            n = len(images)
            n_train = int(n * train_ratio)
            n_val = int(n * (train_ratio + val_ratio))

            if mode == 'train':
                selected = images[:n_train]
            elif mode == 'val':
                selected = images[n_train:n_val]
            else:
                selected = images[n_val:]

            for f in selected:
                self.image_paths.append(os.path.join(domain_path, f))
                self.labels.append(self.domain_to_label[domain])

        tfms = [transforms.Resize((img_size, img_size))]
        if mode == 'train' and flip_prob > 0:
            tfms.append(transforms.RandomHorizontalFlip(p=flip_prob))
        tfms += [
            transforms.ToTensor(),
            transforms.Normalize([0.5] * 3, [0.5] * 3),
        ]
        self.transform = transforms.Compose(tfms)

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        path = self.image_paths[idx]
        img = Image.open(path).convert('RGB')
        img = self.transform(img)
        return img, self.labels[idx], path


class DomainSubset(RetinalDataset):
    """RetinalDataset filtered down to a single domain (by folder name)."""

    def __init__(self, root_dir, domain_name, img_size=64, mode='train',
                 train_ratio=0.8, val_ratio=0.1, flip_prob=0.3, seed=42):
        super().__init__(root_dir, img_size, mode, train_ratio, val_ratio, flip_prob, seed)
        keep = [i for i, l in enumerate(self.labels) if self.label_to_domain[l] == domain_name]
        self.image_paths = [self.image_paths[i] for i in keep]
        self.labels = [self.labels[i] for i in keep]


class PairedDataset:
    """
    Pairs two DomainSubsets by index position (image i of domain A with image i of domain B).

    IMPORTANT: these are NOT the same patient/scene - true paired fundus data (same eye,
    healthy and diseased) essentially does not exist. This positional pairing is what makes
    Pix2Pix usable at all in an unpaired-data setting, and it is also *why* Pix2Pix performs
    poorly (its L1 loss penalizes any spatial mismatch between unrelated images).
    """

    def __init__(self, root_dir, source_domain, target_domain, img_size=64, mode='train',
                 train_ratio=0.8, val_ratio=0.1, flip_prob=0.3, seed=42):
        self.source = DomainSubset(root_dir, source_domain, img_size, mode,
                                    train_ratio, val_ratio, flip_prob, seed)
        self.target = DomainSubset(root_dir, target_domain, img_size, mode,
                                    train_ratio, val_ratio, flip_prob, seed)
        self.length = min(len(self.source), len(self.target))

    def __len__(self):
        return self.length

    def __getitem__(self, idx):
        src_img, _, _ = self.source[idx]
        tgt_img, _, _ = self.target[idx]
        return src_img, tgt_img


# ============================================================
# Model
# ============================================================

class Generator(nn.Module):
    """Encoder-decoder with additive skip connections, matching the original training run."""

    def __init__(self, in_channels=3, out_channels=3):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, 64, 4, stride=2, padding=1)
        self.conv2 = nn.Conv2d(64, 128, 4, stride=2, padding=1)
        self.conv3 = nn.Conv2d(128, 256, 4, stride=2, padding=1)
        self.conv4 = nn.Conv2d(256, 512, 4, stride=2, padding=1)

        self.deconv1 = nn.ConvTranspose2d(512, 256, 4, stride=2, padding=1)
        self.deconv2 = nn.ConvTranspose2d(256, 128, 4, stride=2, padding=1)
        self.deconv3 = nn.ConvTranspose2d(128, 64, 4, stride=2, padding=1)
        self.deconv4 = nn.ConvTranspose2d(64, out_channels, 4, stride=2, padding=1)

        self.bn1, self.bn2, self.bn3 = nn.BatchNorm2d(128), nn.BatchNorm2d(256), nn.BatchNorm2d(512)
        self.bn4, self.bn5, self.bn6 = nn.BatchNorm2d(256), nn.BatchNorm2d(128), nn.BatchNorm2d(64)

        self.relu = nn.ReLU(inplace=True)
        self.leaky = nn.LeakyReLU(0.2, inplace=True)
        self.tanh = nn.Tanh()

    def forward(self, x):
        e1 = self.leaky(self.conv1(x))
        e2 = self.leaky(self.bn1(self.conv2(e1)))
        e3 = self.leaky(self.bn2(self.conv3(e2)))
        e4 = self.leaky(self.bn3(self.conv4(e3)))

        d1 = self.relu(self.bn4(self.deconv1(e4))) + e3
        d2 = self.relu(self.bn5(self.deconv2(d1))) + e2
        d3 = self.relu(self.bn6(self.deconv3(d2))) + e1
        return self.tanh(self.deconv4(d3))


class Discriminator(nn.Module):
    """PatchGAN, conditioned on the source image."""

    def __init__(self, in_channels=3):
        super().__init__()
        self.model = nn.Sequential(
            nn.Conv2d(in_channels * 2, 64, 4, stride=2, padding=1),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(64, 128, 4, stride=2, padding=1),
            nn.BatchNorm2d(128), nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(128, 256, 4, stride=2, padding=1),
            nn.BatchNorm2d(256), nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(256, 512, 4, stride=2, padding=1),
            nn.BatchNorm2d(512), nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(512, 1, 4, padding=1),
        )

    def forward(self, img_a, img_b):
        return self.model(torch.cat([img_a, img_b], dim=1))


def weights_init(m):
    name = m.__class__.__name__
    if 'Conv' in name:
        nn.init.normal_(m.weight.data, 0.0, 0.02)
        if m.bias is not None:
            nn.init.constant_(m.bias.data, 0)
    elif 'BatchNorm' in name:
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0)


def denorm(t):
    return (t.clamp(-1, 1) + 1) / 2


def train(args):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    os.makedirs(CHECKPOINT_DIR, exist_ok=True)

    train_ds = PairedDataset(DATASET_PATH, SOURCE_DOMAIN, TARGET_DOMAIN, IMG_SIZE, 'train',
                              TRAIN_RATIO, VAL_RATIO, FLIP_PROB, SEED)
    loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)

    G, D = Generator().to(device), Discriminator().to(device)
    G.apply(weights_init)
    D.apply(weights_init)

    opt_g = optim.Adam(G.parameters(), lr=LR, betas=(0.5, 0.999))
    opt_d = optim.Adam(D.parameters(), lr=LR, betas=(0.5, 0.999))
    criterion_gan = nn.MSELoss()
    criterion_l1 = nn.L1Loss()

    with torch.no_grad():
        dummy = torch.randn(1, 3, IMG_SIZE, IMG_SIZE).to(device)
        patch_size = D(dummy, dummy).shape[2]

    start_epoch = 0
    latest_path = os.path.join(CHECKPOINT_DIR, 'latest.pth')
    if args.resume and os.path.exists(latest_path):
        ckpt = torch.load(latest_path, map_location=device)
        G.load_state_dict(ckpt['G'])
        D.load_state_dict(ckpt['D'])
        opt_g.load_state_dict(ckpt['opt_g'])
        opt_d.load_state_dict(ckpt['opt_d'])
        start_epoch = ckpt['epoch'] + 1
        print(f'Resumed from epoch {start_epoch}')

    for epoch in range(start_epoch, NUM_EPOCHS):
        for step, (source, target) in enumerate(loader):
            source, target = source.to(device), target.to(device)
            b = source.size(0)
            valid = torch.ones(b, 1, patch_size, patch_size, device=device)
            fake_label = torch.zeros(b, 1, patch_size, patch_size, device=device)

            opt_g.zero_grad()
            fake_target = G(source)
            loss_gan = criterion_gan(D(source, fake_target), valid)
            loss_l1 = criterion_l1(fake_target, target) * LAMBDA_L1
            loss_g = loss_gan + loss_l1
            loss_g.backward()
            opt_g.step()

            opt_d.zero_grad()
            loss_d_real = criterion_gan(D(source, target), valid)
            loss_d_fake = criterion_gan(D(source, fake_target.detach()), fake_label)
            loss_d = (loss_d_real + loss_d_fake) / 2
            loss_d.backward()
            opt_d.step()

            if step % 50 == 0:
                print(f'epoch {epoch+1}/{NUM_EPOCHS} step {step}/{len(loader)} '
                      f'g_loss={loss_g.item():.4f} d_loss={loss_d.item():.4f}')

        if (epoch + 1) % SAVE_FREQ == 0 or (epoch + 1) == NUM_EPOCHS:
            torch.save({
                'epoch': epoch, 'G': G.state_dict(), 'D': D.state_dict(),
                'opt_g': opt_g.state_dict(), 'opt_d': opt_d.state_dict(),
            }, latest_path)
            print(f'Saved checkpoint at epoch {epoch+1}')

    print('Pix2Pix training finished.')


def evaluate(args):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    os.makedirs(RESULTS_DIR, exist_ok=True)
    real_dir = os.path.join(RESULTS_DIR, 'real')
    fake_dir = os.path.join(RESULTS_DIR, 'fake')
    os.makedirs(real_dir, exist_ok=True)
    os.makedirs(fake_dir, exist_ok=True)

    ckpt = torch.load(args.checkpoint, map_location=device)
    G = Generator().to(device)
    G.load_state_dict(ckpt['G'])
    G.eval()

    # Evaluate on the source-domain test split; target images are used only as the
    # (positionally-paired) reference for PSNR/SSIM/MSE.
    test_ds = PairedDataset(DATASET_PATH, SOURCE_DOMAIN, TARGET_DOMAIN, IMG_SIZE, 'test',
                             TRAIN_RATIO, VAL_RATIO, flip_prob=0.0, seed=SEED)
    loader = DataLoader(test_ds, batch_size=1, shuffle=False)

    rows = []
    with torch.no_grad():
        for idx, (source, target) in enumerate(loader):
            source, target = source.to(device), target.to(device)
            fake = G(source)

            save_image(denorm(target[0]), os.path.join(real_dir, f'{idx:05d}.png'))
            save_image(denorm(fake[0]), os.path.join(fake_dir, f'{idx:05d}.png'))

            target_np = denorm(target[0]).cpu().numpy().transpose(1, 2, 0)
            fake_np = denorm(fake[0]).cpu().numpy().transpose(1, 2, 0)

            mse = float(np.mean((target_np - fake_np) ** 2))
            psnr_val = sk_psnr(target_np, fake_np, data_range=1.0)
            ssim_val = sk_ssim(target_np, fake_np, data_range=1.0, channel_axis=2)

            rows.append({
                'image_idx': idx, 'source_domain': SOURCE_DOMAIN, 'target_domain': TARGET_DOMAIN,
                'psnr': psnr_val, 'ssim': ssim_val, 'mse': mse,
            })

    csv_path = os.path.join(RESULTS_DIR, 'per_image_metrics.csv')
    with open(csv_path, 'w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=['image_idx', 'source_domain', 'target_domain', 'psnr', 'ssim', 'mse'])
        writer.writeheader()
        writer.writerows(rows)

    print(f'Saved {csv_path}')
    print(f'Real (target) images: {real_dir}  |  Fake images: {fake_dir}')
    print(f'FID: python analysis.py --mode fid --real {real_dir} --fake {fake_dir}')


if __name__ == '__main__':
    p = argparse.ArgumentParser()
    p.add_argument('--mode', choices=['train', 'evaluate'], required=True)
    p.add_argument('--resume', action='store_true')
    p.add_argument('--checkpoint', type=str, default=os.path.join(CHECKPOINT_DIR, 'latest.pth'))
    args = p.parse_args()

    if args.mode == 'train':
        train(args)
    else:
        evaluate(args)


Writing pix2pix.py


In [ ]:
# (path, expected top-level key in the wrapped checkpoint dict)
existing_checkpoints = {
    'DCGAN':    ('/content/drive/MyDrive/CSE720/dcgan_results/generator_final.pth', 'G'),
    'CycleGAN': ('/content/drive/MyDrive/CSE720/cyclegan_results/generator_AtoB_final.pth', 'G_AtoB'),
    'Pix2Pix':  ('/content/drive/MyDrive/CSE720/pix2pix_results/generator_final.pth', 'G'),
}

# Where each script's own `--checkpoint` default (CHECKPOINT_DIR + 'latest.pth') looks.
adapted_checkpoint_paths = {
    'DCGAN':    os.path.join(cfg.base_dir, 'checkpoints', 'dcgan', 'latest.pth'),
    'CycleGAN': os.path.join(cfg.base_dir, 'checkpoints', 'cyclegan', 'latest.pth'),
    'Pix2Pix':  os.path.join(cfg.base_dir, 'checkpoints', 'pix2pix', 'latest.pth'),
}

for model_name, (old_path, key) in existing_checkpoints.items():
    new_path = adapted_checkpoint_paths[model_name]
    os.makedirs(os.path.dirname(new_path), exist_ok=True)

    if not os.path.exists(old_path):
        print(f"{model_name}: SKIPPED — could not find existing checkpoint at {old_path}")
        continue

    state_dict = torch.load(old_path, map_location='cpu')
    torch.save({key: state_dict}, new_path)
    print(f"{model_name}: wrapped state_dict under key '{key}'\n"
          f"    {old_path}\n -> {new_path}")

DCGAN: wrapped state_dict under key 'G'
    /content/drive/MyDrive/CSE720/dcgan_results/generator_final.pth
 -> /content/drive/MyDrive/CSE720/checkpoints/dcgan/latest.pth
CycleGAN: wrapped state_dict under key 'G_AtoB'
    /content/drive/MyDrive/CSE720/cyclegan_results/generator_AtoB_final.pth
 -> /content/drive/MyDrive/CSE720/checkpoints/cyclegan/latest.pth
Pix2Pix: wrapped state_dict under key 'G'
    /content/drive/MyDrive/CSE720/pix2pix_results/generator_final.pth
 -> /content/drive/MyDrive/CSE720/checkpoints/pix2pix/latest.pth


In [ ]:
!python dcgan.py --mode evaluate \
    --checkpoint /content/drive/MyDrive/CSE720/checkpoints/dcgan/latest.pth

Saved /content/drive/MyDrive/CSE720/results/dcgan/per_image_metrics.csv
Real images: /content/drive/MyDrive/CSE720/results/dcgan/real  |  Fake images: /content/drive/MyDrive/CSE720/results/dcgan/fake
FID: python analysis.py --mode fid --real /content/drive/MyDrive/CSE720/results/dcgan/real --fake /content/drive/MyDrive/CSE720/results/dcgan/fake


In [ ]:
!python cyclegan.py --mode evaluate \
    --checkpoint /content/drive/MyDrive/CSE720/checkpoints/cyclegan/latest.pth

Saved /content/drive/MyDrive/CSE720/results/cyclegan/per_image_metrics.csv
Real images: /content/drive/MyDrive/CSE720/results/cyclegan/real  |  Fake images: /content/drive/MyDrive/CSE720/results/cyclegan/fake
FID: python analysis.py --mode fid --real /content/drive/MyDrive/CSE720/results/cyclegan/real --fake /content/drive/MyDrive/CSE720/results/cyclegan/fake


In [ ]:
!python pix2pix.py --mode evaluate \
    --checkpoint /content/drive/MyDrive/CSE720/checkpoints/pix2pix/latest.pth

Saved /content/drive/MyDrive/CSE720/results/pix2pix/per_image_metrics.csv
Real (target) images: /content/drive/MyDrive/CSE720/results/pix2pix/real  |  Fake images: /content/drive/MyDrive/CSE720/results/pix2pix/fake
FID: python analysis.py --mode fid --real /content/drive/MyDrive/CSE720/results/pix2pix/real --fake /content/drive/MyDrive/CSE720/results/pix2pix/fake


In [ ]:
baseline_results_dirs = {
    'DCGAN':    os.path.join(cfg.base_dir, 'results', 'dcgan'),
    'CycleGAN': os.path.join(cfg.base_dir, 'results', 'cyclegan'),
    'Pix2Pix':  os.path.join(cfg.base_dir, 'results', 'pix2pix'),
}
baseline_csv_paths = {
    model: os.path.join(d, 'per_image_metrics.csv')
    for model, d in baseline_results_dirs.items()
}

missing = [model for model, path in baseline_csv_paths.items() if not os.path.exists(path)]

if missing:
    lines = [f"Missing real evaluation output for: {', '.join(missing)}", "",
              "Check the !python ... --mode evaluate cell output above for errors."]
    raise FileNotFoundError("\n".join(lines))

print("Found real evaluation output for all three baselines:")
for model, path in baseline_csv_paths.items():
    n_rows = len(pd.read_csv(path))
    print(f"  {model:10s} -> {path}  ({n_rows} rows)")

Found real evaluation output for all three baselines:
  DCGAN      -> /content/drive/MyDrive/CSE720/results/dcgan/per_image_metrics.csv  (250 rows)
  CycleGAN   -> /content/drive/MyDrive/CSE720/results/cyclegan/per_image_metrics.csv  (50 rows)
  Pix2Pix    -> /content/drive/MyDrive/CSE720/results/pix2pix/per_image_metrics.csv  (50 rows)


In [ ]:
from scipy import stats
from statsmodels.stats.multitest import multipletests

eyegan_path = os.path.join(cfg.eval_dir, 'EyeGAN_per_item_results.csv')

if not os.path.exists(eyegan_path):
    print(f"Error: Could not find ground truth EyeGAN results at '{eyegan_path}'")
else:
    eyegan_df = pd.read_csv(eyegan_path).replace([np.inf, -np.inf], np.nan)

    stats_rows = []
    metrics = ['psnr', 'ssim', 'mse']

    for model_name, csv_path in baseline_csv_paths.items():
        c_df = pd.read_csv(csv_path).replace([np.inf, -np.inf], np.nan)

        for m in metrics:
            if m in eyegan_df.columns and m in c_df.columns:
                eyegan_vals = eyegan_df[m].dropna().values
                model_vals = c_df[m].dropna().values

                if len(eyegan_vals) > 0 and len(model_vals) > 0:
                    # Welch's t-test (unequal variances)
                    t_stat, p_val = stats.ttest_ind(eyegan_vals, model_vals, equal_var=False)

                    # Cohen's d effect size
                    n1, n2 = len(eyegan_vals), len(model_vals)
                    s1, s2 = np.std(eyegan_vals, ddof=1), np.std(model_vals, ddof=1)
                    s_pooled = np.sqrt(((n1 - 1) * s1**2 + (n2 - 1) * s2**2) / (n1 + n2 - 2))
                    cohens_d = (np.mean(eyegan_vals) - np.mean(model_vals)) / s_pooled if s_pooled > 0 else 0.0

                    stats_rows.append({
                        'metric': m.upper(),
                        'baseline': model_name,
                        'eyegan_mean': np.mean(eyegan_vals),
                        'model_mean': np.mean(model_vals),
                        'welch_t': t_stat,
                        'welch_p': p_val,
                        'cohens_d': cohens_d
                    })

    # FDR Correction (Benjamini-Hochberg)
    stats_df = pd.DataFrame(stats_rows)
    if not stats_df.empty:
        p_vals = stats_df['welch_p'].fillna(1.0).values
        reject, pvals_corrected, _, _ = multipletests(p_vals, method='fdr_bh', alpha=0.05)

        stats_df['welch_p_fdr_corrected'] = pvals_corrected
        stats_df['significant_after_fdr'] = reject

        cols_to_show = ['metric', 'baseline', 'eyegan_mean', 'model_mean', 'welch_p', 'welch_p_fdr_corrected', 'significant_after_fdr', 'cohens_d']

        print("\n--- Baseline-only Statistical Significance Test Results (FDR Corrected) ---")
        print(stats_df[cols_to_show].round(4).to_string(index=False))

        out_path = os.path.join(cfg.eval_dir, 'statistical_significance_results.csv')
        stats_df.to_csv(out_path, index=False)
        print(f"\nSuccessfully saved statistical results to: {out_path}")


--- Baseline-only Statistical Significance Test Results (FDR Corrected) ---
metric baseline  eyegan_mean  model_mean  welch_p  welch_p_fdr_corrected  significant_after_fdr  cohens_d
  PSNR    DCGAN      36.2446     16.2691      0.0                    0.0                   True    6.8015
  SSIM    DCGAN       0.9256      0.4901      0.0                    0.0                   True    5.4080
   MSE    DCGAN       0.0003      0.0319      0.0                    0.0                   True   -3.3136
  PSNR CycleGAN      36.2446     25.2940      0.0                    0.0                   True    3.9418
  SSIM CycleGAN       0.9256      0.8131      0.0                    0.0                   True    3.0682
   MSE CycleGAN       0.0003      0.0047      0.0                    0.0                   True   -3.6614
  PSNR  Pix2Pix      36.2446     15.8726      0.0                    0.0                   True    7.5126
  SSIM  Pix2Pix       0.9256      0.4881      0.0                    0.0   

In [ ]:
out_path = os.path.join(cfg.eval_dir, 'statistical_significance_results.csv')
ablation_dir = cfg.ablation_dir

if not os.path.exists(eyegan_path):
    print(f"Error: Could not find '{eyegan_path}'")
else:
    stats_rows = []
    metrics = ['psnr', 'ssim', 'mse']

    # 1. Process real baselines
    for model_name, csv_path in baseline_csv_paths.items():
        c_df = pd.read_csv(csv_path).replace([np.inf, -np.inf], np.nan)
        for m in metrics:
            if m in eyegan_df.columns and m in c_df.columns:
                eyegan_vals = eyegan_df[m].dropna().values
                model_vals = c_df[m].dropna().values

                if len(eyegan_vals) > 0 and len(model_vals) > 0:
                    t_stat, p_val = stats.ttest_ind(eyegan_vals, model_vals, equal_var=False)
                    n1, n2 = len(eyegan_vals), len(model_vals)
                    s1, s2 = np.std(eyegan_vals, ddof=1), np.std(model_vals, ddof=1)
                    s_pooled = np.sqrt(((n1 - 1) * s1**2 + (n2 - 1) * s2**2) / (n1 + n2 - 2))
                    cohens_d = (np.mean(eyegan_vals) - np.mean(model_vals)) / s_pooled if s_pooled > 0 else 0.0

                    stats_rows.append({
                        'metric': m.upper(), 'baseline': model_name,
                        'eyegan_mean': np.mean(eyegan_vals), 'model_mean': np.mean(model_vals),
                        'welch_t': t_stat, 'welch_p': p_val, 'cohens_d': cohens_d
                    })

    # 2. Process ablation study models
    if os.path.exists(ablation_dir):
        ablation_files = [os.path.join(ablation_dir, f) for f in os.listdir(ablation_dir) if f.endswith('_per_item_results.csv')]
        for file_path in ablation_files:
            model_name = os.path.basename(file_path).replace('_per_item_results.csv', '')
            c_df = pd.read_csv(file_path).replace([np.inf, -np.inf], np.nan)

            for m in metrics:
                if m in eyegan_df.columns and m in c_df.columns:
                    eyegan_vals = eyegan_df[m].dropna().values
                    model_vals = c_df[m].dropna().values

                    if len(eyegan_vals) > 0 and len(model_vals) > 0:
                        t_stat, p_val = stats.ttest_ind(eyegan_vals, model_vals, equal_var=False)
                        n1, n2 = len(eyegan_vals), len(model_vals)
                        s1, s2 = np.std(eyegan_vals, ddof=1), np.std(model_vals, ddof=1)
                        s_pooled = np.sqrt(((n1 - 1) * s1**2 + (n2 - 1) * s2**2) / (n1 + n2 - 2))
                        cohens_d = (np.mean(eyegan_vals) - np.mean(model_vals)) / s_pooled if s_pooled > 0 else 0.0

                        stats_rows.append({
                            'metric': m.upper(), 'baseline': model_name,
                            'eyegan_mean': np.mean(eyegan_vals), 'model_mean': np.mean(model_vals),
                            'welch_t': t_stat, 'welch_p': p_val, 'cohens_d': cohens_d
                        })

    # 3. Apply FDR correction
    stats_df = pd.DataFrame(stats_rows)
    if not stats_df.empty:
        p_vals = stats_df['welch_p'].fillna(1.0).values
        reject, pvals_corrected, _, _ = multipletests(p_vals, method='fdr_bh', alpha=0.05)

        stats_df['welch_p_fdr_corrected'] = pvals_corrected
        stats_df['significant_after_fdr'] = reject

        cols_to_show = ['metric', 'baseline', 'eyegan_mean', 'model_mean', 'welch_p_fdr_corrected', 'significant_after_fdr', 'cohens_d']

        print("\n--- Complete Combined Statistical Test Results (Real Baselines + Ablations) ---")
        print(stats_df[cols_to_show].round(4).to_string(index=False))

        stats_df.to_csv(out_path, index=False)
        print(f"\nSaved combined statistical results to: {out_path}")


--- Complete Combined Statistical Test Results (Real Baselines + Ablations) ---
metric             baseline  eyegan_mean  model_mean  welch_p_fdr_corrected  significant_after_fdr  cohens_d
  PSNR                DCGAN      36.2446     16.2691                 0.0000                   True    6.8015
  SSIM                DCGAN       0.9256      0.4901                 0.0000                   True    5.4080
   MSE                DCGAN       0.0003      0.0319                 0.0000                   True   -3.3136
  PSNR             CycleGAN      36.2446     25.2940                 0.0000                   True    3.9418
  SSIM             CycleGAN       0.9256      0.8131                 0.0000                   True    3.0682
   MSE             CycleGAN       0.0003      0.0047                 0.0000                   True   -3.6614
  PSNR              Pix2Pix      36.2446     15.8726                 0.0000                   True    7.5126
  SSIM              Pix2Pix       0.9256      0